# Introduction debug syntax_deriver_db

# Add root to sys.path

In [1]:
import os
import sys

# Add the root directory to the Python path
module_dir = os.path.abspath('..')
if module_dir not in sys.path:
    sys.path.append(module_dir)
for x in sys.path:
    print(x)

/opt/anaconda3/envs/python312/lib/python312.zip
/opt/anaconda3/envs/python312/lib/python3.12
/opt/anaconda3/envs/python312/lib/python3.12/lib-dynload

/opt/anaconda3/envs/python312/lib/python3.12/site-packages
/opt/anaconda3/envs/python312/lib/python3.12/site-packages/setuptools/_vendor
/Users/hale/PycharmProjects/MathAssertGPT


# Imports

In [2]:
import torch
from pathlib import Path

import pandas as pd

from source.shared import Encoder
from source.shared import load_model

from source.create_model import create_model

from Settings import Settings

!python --version
!which python

Python 3.12.8
/opt/anaconda3/envs/python312/bin/python


# Settings

In [3]:
import panel as pn
pn.extension()
pn.config.sizing_mode="stretch_width"

settings = Settings()
settings.view()

WidgetBox(max_width=600, sizing_mode='stretch_width')
    [0] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='Limit count', sizing_mode='stretch_width', value=40000)
    [2] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntInput(name='n_embd', sizing_mode='stretch_width', value=1000)
        [2] IntInput(name='n_head', sizing_mode='stretch_width', value=10)
        [3] IntInput(name='block_size', sizing_mode='stretch_width', value=150)
        [4] FloatInput(name='dropout', sizing_mode='stretch_width', value=0.2)
        [5] IntInput(name='n_layer', sizing_mode='stretch_width', value=10)
    [3] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] FloatInput(name='learning_rate', sizing_mode='stretch_width', value=0.0001)
    [4] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] LiteralInput(name='mmx_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [2] LiteralInput(name='corpus01_file_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [3] LiteralInput(name='corpus_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
        [4] LiteralInput(name='model_folder_path', sizing_mode='stretch_width', value='/Users/hale/PycharmProjec...)
    [5] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')

# Load model

In [4]:
def set_up_model() -> Path:
    model_folder_path = Path(settings.model_folder_path)
    model_name = 'model.pt'
    model_file_path = model_folder_path.joinpath(model_name).resolve()
    if model_file_path.exists():
        print(f"model_file already exists: {model_file_path}")
    else:
        create_model(settings=settings)
    return model_file_path

model_file_path = set_up_model()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.backends.mps.is_available():
    device = "mps"
print(f'device={device}')
encoder = Encoder.load_from_json(corpus_folder_path=settings.corpus_folder_path)
print(f'loading model and optimizer from checkpoint={model_file_path}')
model, optimizer = load_model(model_checkpoint_path=model_file_path, device=device, encoder=encoder)

model_file already exists: /Users/hale/PycharmProjects/MathAssertGPT/model/model.pt
device=mps
loading model and optimizer from checkpoint=/Users/hale/PycharmProjects/MathAssertGPT/model/model.pt


# Create an example

In [5]:
from source.evaluate_model import ModelEvaluator

def make_syntax_deriver_db(max_examples: int):
    model_evaluator = ModelEvaluator(corpus_folder_path=settings.corpus_folder_path, model=model)
    model_evaluator.evaluate_model(max_examples=max_examples)
    syntax_deriver_db = model_evaluator.syntax_deriver.syntax_deriver_db
    return syntax_deriver_db

max_examples = 1
syntax_deriver_db = make_syntax_deriver_db(max_examples=max_examples)

vocab_size=160
epoch=100; step=1000; n_head=10; n_layer=10
=== start evaluate_model ===
max_val_examples=1


In [6]:
conn = syntax_deriver_db.conn
cursor = conn.cursor()
sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
cursor.execute(sql)
math_statement_rows = cursor.fetchall()
example_count = len(math_statement_rows)
print(f'Example count: {example_count}')

Example count: 1


In [7]:
math_statement_row = math_statement_rows[0]
print(math_statement_row)

Row(id=1, statement='( ( ( F B -> A e. x ps ) -> E. A ) -> A }', context='|- \n|- ( ( ( F B -> A e. x ps ) -> E. A ) -> A } <|over|>', derivation=None, derivation_correct_count=5, syntax_deriver_error='SyntaxDeriverWffRuleError')


In [8]:
derivation = math_statement_row.derivation
derivation_correct_count = math_statement_row.derivation_correct_count
statement_id = math_statement_row.id
context = math_statement_row.context
syntax_deriver_error = math_statement_row.syntax_deriver_error
wff_statement = math_statement_row.statement

print(f'statement_id: {statement_id}')
print(f'wff_statement: {wff_statement}')
print(f'derivation: {derivation}')
print(f'derivation_correct_count: {derivation_correct_count}')
print(f'syntax_deriver_error: {syntax_deriver_error}')
print(f'context:\n{context}')

statement_id: 1
wff_statement: ( ( ( F B -> A e. x ps ) -> E. A ) -> A }
derivation: None
derivation_correct_count: 5
syntax_deriver_error: SyntaxDeriverWffRuleError
context:
|- 
|- ( ( ( F B -> A e. x ps ) -> E. A ) -> A } <|over|>


# Query

In [9]:
def beta(query):
    df = pd.read_sql_query(query, conn)
    return df

def phi(query, conn):
    df = pd.read_sql_query(query, conn)
    return pn.pane.DataFrame(df)

In [10]:
phi("SELECT * FROM math_statements", conn)

DataFrame(DataFrame, sizing_mode='stretch_width')

In [11]:
phi("SELECT * FROM rule_errors", conn)

DataFrame(DataFrame, sizing_mode='stretch_width')

In [12]:
def colorize(text, color):
    print(f'colorize text={text}')
    return f'<span style="color:{color}">{text}</span>'

dynamic_container = pn.Column(margin=(0, 0, 0, 0))

conn = syntax_deriver_db.conn
cursor = conn.cursor()
sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
cursor.execute(sql)
math_statement_rows = cursor.fetchall()
for math_statement_index in range(len(math_statement_rows)):
    math_statement_row = math_statement_rows[math_statement_index]
    statement_id = math_statement_row.id
    statement = math_statement_row.statement
    context = math_statement_row.context
    derivation = math_statement_row.derivation
    derivation_correct_count = math_statement_row.derivation_correct_count
    syntax_deriver_error = math_statement_row.syntax_deriver_error
    prompt = context.split('\n')[0]
    lines = []
    lines.append(f'prompt: {prompt}')
    lines.append(f'predicted_statement: {statement}')
    lines.append(f'error: {syntax_deriver_error}')
    lines.append(f'derivation_correct_count={derivation_correct_count}')
    dynamic_container.append(pn.pane.Str('\n'.join(lines)))
    print(f'statement: {statement}')

    if derivation is None:
        lines = []
        lines.append(f'--- Possible continuations ---')
        dynamic_container.append(pn.pane.Str('\n'.join(lines)))
        sql = f'SELECT statement_id, rule_name, rule, mark_index, rule_tokens, current_rule_tokens FROM rule_errors WHERE statement_id = {statement_id} ORDER BY id'
        cursor.execute(sql)
        rule_error_rows = cursor.fetchall()
        for i in range(len(rule_error_rows)):
            lines = []
            rule_error_row = rule_error_rows[i]
            rule_tokens = rule_error_row.rule_tokens
            rule_name = rule_error_row.rule_name
            rule = rule_error_row.rule
            mark_index = rule_error_row.mark_index
            print(f'derivation_correct_count={derivation_correct_count}')
            print(f'mark_index: {mark_index}')
            token_index = derivation_correct_count
            print(f'token_index={token_index}')
            current_rule_tokens = rule_error_row.current_rule_tokens
            print(f'current_rule_tokens={current_rule_tokens}')
            current_token_index = max(0, derivation_correct_count + mark_index)
            print(f'current_token_index={current_token_index}')
            statement_split = statement.split()
            rule_split = rule.split()
            accumulated = " ".join(statement_split[:token_index]) + " "
            peeked = " ".join(statement_split[token_index: current_token_index]) + " "
            # current_token = statement[current_token_index]
            # current_token = f'{statement[current_token_index]} '
            current_token = " ".join(statement_split[current_token_index: current_token_index+1]) + " "
            rest = " ".join(statement_split[current_token_index+1:])
            colorize_text = f'{colorize(text=accumulated, color="blue")}{colorize(text=peeked, color="#00BB00")}{colorize(current_token, "red")}{colorize(text=rest, color="black")}'
            print(colorize_text)
            print(f'accumulated: {accumulated}')
            print(f'current_token: {current_token}')
            print(f'peeked: {peeked}')
            print(f'rest: {rest}')
            lines.append(f'Rule {rule_name}: {rule} mark_index={mark_index}')
            lines.append(f'Expected: {rule_split[mark_index]}')
            subpanel = pn.Column(
                # pn.pane.Str(f'===== Example {i + 1} error: ? ====='),
                pn.pane.Str('\n'.join(lines)),
                pn.pane.Markdown(
                    # Metamath color is Color(red: 0.933, green: 1, blue: 0.98, alpha: 1) red: EF green: FF blue: FB '#effffb'
                    colorize_text, styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb'}
                )
            )
            dynamic_container.append(subpanel)

statement: ( ( ( F B -> A e. x ps ) -> E. A ) -> A }
derivation_correct_count=5
mark_index: 0
token_index=5
current_rule_tokens=-> A e. x ps ) -> E. A ) -> A }
current_token_index=5
colorize text=( ( ( F B 
colorize text= 
colorize text=-> 
colorize text=A e. x ps ) -> E. A ) -> A }
<span style="color:blue">( ( ( F B </span><span style="color:#00BB00"> </span><span style="color:red">-> </span><span style="color:black">A e. x ps ) -> E. A ) -> A }</span>
accumulated: ( ( ( F B 
current_token: -> 
peeked:  
rest: A e. x ps ) -> E. A ) -> A }
derivation_correct_count=5
mark_index: 0
token_index=5
current_rule_tokens=-> A e. x ps ) -> E. A ) -> A }
current_token_index=5
colorize text=( ( ( F B 
colorize text= 
colorize text=-> 
colorize text=A e. x ps ) -> E. A ) -> A }
<span style="color:blue">( ( ( F B </span><span style="color:#00BB00"> </span><span style="color:red">-> </span><span style="color:black">A e. x ps ) -> E. A ) -> A }</span>
accumulated: ( ( ( F B 
current_token: -> 
peeked

In [13]:
outer_style = {
    # 'background': 'black',
    'border': '0px solid black',
    'padding': '5px',
    'margin': "0px",
}

pn.Column (
    dynamic_container,
    styles=outer_style,
)

Column(sizing_mode='stretch_width', styles={'border': '0px solid blac...})
    [0] Column(margin=(0, 0, 0, 0), sizing_mode='stretch_width')
        [0] Str(str, sizing_mode='stretch_width')
        [1] Str(str, sizing_mode='stretch_width')
        [2] Column(sizing_mode='stretch_width')
            [0] Str(str, sizing_mode='stretch_width')
            [1] Markdown(str, sizing_mode='stretch_width', styles={'font-family': 'monospace...})
        [3] Column(sizing_mode='stretch_width')
            [0] Str(str, sizing_mode='stretch_width')
            [1] Markdown(str, sizing_mode='stretch_width', styles={'font-family': 'monospace...})